#  Projet Data Engineering : Géocodage des Hôtels

Après avoir récupéré la liste des hôtels via Scrapy, il nous manque une information cruciale pour la carte : **les coordonnées GPS (Latitude, Longitude)**.

Dans ce carnet, nous allons utiliser une API gratuite et open-source : **Nominatim (OpenStreetMap)**.

###  Objectifs
1.  **Interroger une API REST** avec la librairie `requests`.
2.  **Respecter les quotas** (Rate Limiting) pour ne pas se faire bannir.
3.  **Manipuler des DataFrames** Pandas pour enrichir des données existantes.
4.  **Gérer les erreurs** (Hôtel introuvable -> Fallback sur la ville).

## 1. Importation des Librairies

Nous avons besoin de plusieurs outils :
- **pandas** : Pour lire notre fichier CSV d'hôtels et le modifier.
- **requests** : Pour envoyer des demandes (requêtes HTTP) à OpenStreetMap.
- **time** : Indispensable pour faire des pauses (`sleep`) entre les requêtes et respecter les règles d'utilisation.
- **os** : Pour vérifier si nos fichiers existent sur l'ordinateur.

In [1]:
import pandas as pd
import requests
import time
import os

## 2. Configuration des Fichiers

Nous définissons ici les chemins d'entrée (le fichier produit par Scrapy) et de sortie (le fichier final avec GPS).

In [2]:
# Nom du fichier CSV généré par l'étape précédente (Scrapy)
INPUT_FILE = "booking_full_data.csv"

# Nom du fichier final qui contiendra les coordonnées GPS
OUTPUT_FILE = "booking_final_gps.csv"

## 3. Configuration de l'API Nominatim (User-Agent)

 **Point Critique** : Nominatim est un service gratuit maintenu par des bénévoles. Il impose des règles strictes :
1.  Maximum **1 requête par seconde**.
2.  Obligation de s'identifier via un **User-Agent** (votre "carte d'identité" numérique).

Risque de banissement si vous envoyez trop de requêtes (Erreur 403).

In [3]:
# URL de base de l'API de recherche Nominatim
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

# En-têtes de la requête HTTP (Headers)
# Il est impératif de mettre un email ou un identifiant de projet valide.
HEADERS = {
    'User-Agent': 'Projet_Etudiant_Data_Scraping_Booking_v1 (mon_email@ecole.fr)' 
}

## 4. Création de la Fonction de Géocodage

Cette fonction `get_gps_safe` est notre outil principal. Elle :
1.  Reçoit une adresse ou un lieu (`query`).
2.  Attend 1.1 seconde (pour être sûr d'être > 1 sec).
3.  Envoie la requête à l'API.
4.  Renvoie la Latitude et la Longitude si tout va bien, ou `None, None` sinon.

In [4]:
def get_gps_safe(query):
    """
    Interroge Nominatim pour obtenir les coordonnées GPS d'un lieu.
    Respecte le délai d'attente imposé.
    """
    # Paramètres de l'URL (ex: ?q=Paris&format=json&limit=1)
    params = {
        'q': query,
        'format': 'json',
        'limit': 1 # On ne veut que le meilleur résultat
    }
    
    try:
        # --- RÈGLE D'OR : Pause avant chaque requête ---
        time.sleep(1.1) 
        
        # Envoi de la requête GET
        response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
        
        # Si la requête a réussi (Code 200)
        if response.status_code == 200:
            data = response.json()
            # Si l'API a trouvé au moins un résultat
            if data:
                return data[0]['lat'], data[0]['lon']
            
        # Si on est banni (Code 403)
        elif response.status_code == 403:
            print(" ERREUR 403 : User-Agent banni ou requêtes trop rapides !")
            return None, None
            
    except Exception as e:
        # En cas de coupure internet ou autre bug
        print(f" Erreur réseau : {e}")
    
    # Si rien trouvé ou erreur, on renvoie vide
    return None, None

## 5. Chargement des Données

Nous chargeons le fichier CSV produit par le robot Scrapy. Si le fichier n'existe pas, on arrête tout.

In [5]:
if not os.path.exists(INPUT_FILE):
    print(f"❌ ERREUR : Le fichier '{INPUT_FILE}' est introuvable. Avez-vous lancé le Scrapy ?")
else:
    # Chargement dans un DataFrame Pandas
    df = pd.read_csv(INPUT_FILE)
    print(f" Chargement réussi : {len(df)} hôtels à traiter.")

 Chargement réussi : 675 hôtels à traiter.


## 6. Préparation des Colonnes

Nous ajoutons deux nouvelles colonnes `latitude` et `longitude` à notre tableau, initialement vides (`None`). Elles seront remplies au fur et à mesure.

In [6]:
# On vérifie si les colonnes existent déjà, sinon on les crée
if 'latitude' not in df.columns:
    df['latitude'] = None
    df['longitude'] = None

# Aperçu des 5 premières lignes
df.head()

,id_ville,ville,nom_hotel,url,score,prix,description,latitude,longitude
0,1,Mont Saint Michel,La ferme de la begossiere,https://www.booking.com/hotel/fr/la-ferme-de-l...,Avec une note de 8.6,127,L’établissement La ferme de la begossiere se t...,None,None
1,2,St Malo,Ibis Budget Saint Malo Centre,https://www.booking.com/hotel/fr/etap-saint-ma...,Avec une note de 7.9,52,Situé à 25 minutes à pied de la plage de Saint...,None,None
2,3,Bayeux,Bel appartement centre ville Bayeux avec place...,https://www.booking.com/hotel/fr/bel-apparteme...,Avec une note de 8.0,89,L’hébergement Bel appartement centre ville Bay...,None,None
3,4,Le Havre,Hôtel Spa Vent d'Ouest,https://www.booking.com/hotel/fr/vent-d-ouest-...,Avec une note de 8.5,135,"L’Hôtel Spa Vent d’Ouest se situe au Havre, à ...",None,None
4,5,Rouen,Holiday Inn Express - Rouen Centre - Rive Gauc...,https://www.booking.com/hotel/fr/holiday-inn-e...,Avec une note de 8.5,77,"Situé à Rouen, l’établissement Holiday Inn Exp...",None,None


## 7. Boucle Principale de Géocodage

C'est le cœur du script. Nous allons parcourir chaque hôtel un par un.

**Stratégie "Fallback" (Plan B)** :
1.  On essaie de trouver l'hôtel avec son **Nom + Ville** (ex: "Hôtel de la Paix, Paris").
2.  Si ça échoue (introuvable), on cherche juste la **Ville** (ex: "Paris, France"). Cela permet d'avoir au moins un point sur la carte, même s'il est moins précis.

⏳ *Note : Ce processus peut être long (environ 1.5 seconde par hôtel).* 

In [7]:
print("🌍 Démarrage du géocodage respectueux (1 req / 1.1 sec)...")

# iterrows() permet de parcourir le DataFrame ligne par ligne
for index, row in df.iterrows():
    
    # OPTIMISATION : Si on a déjà le GPS (en cas de relance du script), on passe au suivant
    if pd.notna(row['latitude']):
        continue

    # Affichage de progression
    print(f"   [{index+1}/{len(df)}] Géocodage : {row['nom_hotel']} ({row['ville']})... ", end="")
    
    # --- ESSAI 1 : Recherche Précise (Nom + Ville) ---
    lat, lon = get_gps_safe(f"{row['nom_hotel']}, {row['ville']}")
    
    # --- ESSAI 2 : Fallback Ville (Plan B) ---
    # Si l'essai 1 a échoué (lat est None), on cherche juste la ville
    if not lat:
        print(" -> Fallback Ville... ", end="")
        lat, lon = get_gps_safe(f"{row['ville']}, France")

    # Enregistrement du résultat dans le DataFrame
    if lat:
        df.at[index, 'latitude'] = lat
        df.at[index, 'longitude'] = lon
        print("✅ OK")
    else:
        print(" Échec total")

    # --- SAUVEGARDE DE SÉCURITÉ ---
    # Toutes les 10 lignes, on sauvegarde le fichier.
    # Si le PC plante, on ne perdra pas tout le travail !
    if index % 10 == 0:
        df.to_csv(OUTPUT_FILE, index=False)

🌍 Démarrage du géocodage respectueux (1 req / 1.1 sec)...
   [1/675] Géocodage : La ferme de la begossiere (Mont Saint Michel)...  -> Fallback Ville... ✅ OK
   [2/675] Géocodage : Ibis Budget Saint Malo Centre (St Malo)... ✅ OK
   [3/675] Géocodage : Bel appartement centre ville Bayeux avec place de parking (Bayeux)...  -> Fallback Ville... ✅ OK
   [4/675] Géocodage : Hôtel Spa Vent d'Ouest (Le Havre)...  -> Fallback Ville... ✅ OK
   [5/675] Géocodage : Holiday Inn Express - Rouen Centre - Rive Gauche by IHG (Rouen)...  -> Fallback Ville... ✅ OK
   [6/675] Géocodage : Maison Eugenie (Paris)...  -> Fallback Ville... ✅ OK
   [7/675] Géocodage : Hôtel restaurant du Commerce (Amiens)...  -> Fallback Ville... ✅ OK
   [8/675] Géocodage : Clarance Hotel Lille (Lille)... ✅ OK
   [9/675] Géocodage : Hotel La Diligence (Strasbourg)...  -> Fallback Ville... ✅ OK
   [10/675] Géocodage : Pour les amoureux de ballades dans les Vosges - 4 nuits minimum (Chateau du Haut Koenigsbourg)...  -> Fallback V

## 8. Nettoyage et Sauvegarde Finale

Une fois la boucle terminée, on fait un dernier nettoyage (suppression des lignes sans GPS s'il y en a) et on sauvegarde le résultat final.

In [8]:
# On supprime les lignes qui n'ont toujours pas de latitude (échecs complets)
df_clean = df.dropna(subset=['latitude'])

# Sauvegarde finale sans l'index (0, 1, 2...) qui est inutile dans le CSV
df_clean.to_csv(OUTPUT_FILE, index=False)

print(f"\n Terminé ! Fichier final généré : '{OUTPUT_FILE}'")
print(f"Nombre de lignes avec GPS : {len(df_clean)}")


 Terminé ! Fichier final généré : 'booking_final_gps.csv'
Nombre de lignes avec GPS : 675


## 📚 Glossaire Technique & Commandes

| Fonction / Commande | Paramètres Clés | Définition Simple |
| :--- | :--- | :--- |
| **`requests.get`** | `url`, `params`, `headers` | Envoie une demande d'information à un serveur web (API). C'est comme taper une adresse dans un navigateur. |
| **`response.json()`** | *aucun* | Transforme la réponse textuelle du serveur (format JSON) en dictionnaire Python utilisable. |
| **`time.sleep`** | `seconds` | Met le programme en pause. Crucial pour ne pas surcharger les serveurs gratuits comme Nominatim. |
| **`pd.read_csv`** | `filepath` | Charge un fichier CSV (Excel simplifié) dans un tableau Python (DataFrame). |
| **`df.iterrows()`** | *aucun* | Permet de faire une boucle `for` sur chaque ligne du tableau. |
| **`df.at[index, col]`** | `index`, `colonne` | Permet d'écrire une valeur précise dans une case du tableau (coordonnées ligne/colonne). |
| **`df.to_csv`** | `filename`, `index=False` | Sauvegarde le tableau Python dans un fichier sur le disque dur. |
| **`User-Agent`** | *dans headers* | Une chaîne de texte qui dit au serveur "Qui je suis". Obligatoire pour beaucoup d'API. |